In [9]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

In [10]:
from PIL import Image
import io
import base64


def compress_image(image_path, max_size=(800, 800)):
    """压缩图片到合适大小"""
    with Image.open(image_path) as img:
        img.thumbnail(max_size)

        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=85)

        return base64.b64encode(buffer.getvalue()).decode('utf-8')


base64_image = compress_image("aaa.jpg")
print(f"压缩后大小: {len(base64_image) / 1024:.2f} KB")  # 确保 <500KB

压缩后大小: 96.72 KB


In [11]:

messages = [{
    "role": "user",
    "content": [
        {"type": "text", "text": "请将这个图片的内容尽可能全部提取出来"},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
    ]
}]

response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages
)
print(response.choices[0].message.content)

以下是图片中的内容提取：

发票抬头：
北京市鑫裕汽车用品旗舰店  
纳税人识别号：9141010037748287YM  
地址、电话：北京市通州区高井文化产业园8号 010-82354236  
开户行及账号：工商银行北京燕京支行 020201128562436333  

发票编号：
No 58971153  
机器编号：928805152312  
发票代码：1100213130  
开票日期：2021年06月06日  

购买方：
名称： 北京车无忧汽车修理服务有限公司  
纳税人识别号：91110112D42632563  
地址、电话： 北京市通州区新华南路85号 010-82351310  
开户行及账号：工商银行北京市通州区新华南路支行 0202011509102435  

货物或应税劳务、服务名称 规格型号 单位 数量 单价 金额 税率 税额  
- 洗涤剂*汽车泡沫清洗剂 件 10 16.00 160.00 13% 20.80  
- 日用杂品*汽车美容抛光用品 套 5 150.00 750.00 13% 97.50  
- 日用杂品*汽车清洗用品 套 5 60.00 300.00 13% 39.00  

合计金额（小写）：￥1,367.30  
合计金额（大写）：壹仟叁佰陆拾柒圆叁角贰分  
合计税额：￥157.30  

收款人：悍也  
复核：好懂  
开票人：砚益  

发票印章：
- 北京增值税普通发票（红色圆章）
- 北京鑫裕汽车用品旗舰店 专用章 9141010037748287YM  

备注栏含乱码字符，不作提取。

以上为图片中主要内容的完整提取。


In [14]:
prompt="""
请按照如下格式解析图片数据,可能会有一些模糊的内容你需要仔细查看，对于图片中的中文内容你需要仔细解析
**发票头部信息：**
- 发票代码
- 机器编号
- 发票名称
- 发票号码
- 开票日期

**购买方信息：**
- 名 称
- 纳税人识别号
- 地址、电话
- 开户行及账号

**密码区（密文）：**


**货物或应税劳务、服务明细：**

| 货物或应税劳务、服务名称 | 规格型号 | 单位 | 数量 | 单价 | 金额 | 税率 | 税额 |


**合计：**
- 金额
- 税额

**价税合计：**
- （大写
- （小写）

**销售方信息：**
- 名 称：
- 纳税人识别号：
- 地址、电话：
- 开户行及账号：

**签章及经办人信息：**
- 收款人：
- 复核：
- 开票人：
- 销售方：

**备注栏：**

**右侧边栏：**
第三联

**其他标识：**
- 右上角重复打印
- 销售方发票专用章内含税号
"""
messages = [{
    "role": "user",
    "content": [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
    ]
}]

response = client.chat.completions.create(
    model="openai/gpt-4.1-mini",
    messages=messages
)
print(response.choices[0].message.content)

**发票头部信息：**
- 发票代码：1100213130
- 机器编号：928805152312
- 发票名称：增值税专用发票
- 发票号码：58971153
- 开票日期：2021年06月06日

**购买方信息：**
- 名 称：北京车无忧汽车修理服务有限公司
- 纳税人识别号：91110112X04632563
- 地址、电话：北京市通州区新华南路85号 010-82351310
- 开户行及账号：工商银行北京通州区新华南路支行 0202011509102435

**密码区（密文）：** 
>*06 04 / 52 / +90 * +80 99 > 3795 + 67 039 > 7 / 1062 < > * < * 13 + 7> 71 < * +32 4 < / - 21 < + 82018 < 30031 - +55 + 7 >> + * 69 > 90 / 04 / 52 / +90 * + 80 99 >3795 *  

**货物或应税劳务、服务明细：**

| 货物或应税劳务、服务名称          | 规格型号 | 单位 | 数量 | 单价  | 金额  | 税率 | 税额   |
|----------------------------|--------|----|----|-----|-----|----|-------|
| *洗涤剂*汽车泡沫清洗剂             | 件      | 10 | 16.00 | 160.00 | 13% | 20.80  |
| *日用杂品*汽车美容地毯用品           | 套      | 5  | 150.00 | 750.00 | 13% | 97.50  |
| *日用杂品*汽车清洗用品             | 套      | 5  | 60.00  | 300.00 | 13% | 39.00  |

**合计：**
- 金额：1210.00元
- 税额：157.30元

**价税合计：**
- （大写）壹仟贰佰壹拾柒圆叁角零整
- （小写）：¥1367.30

**销售方信息：**
- 名 称：北京威富汽车用品旗舰店
- 纳税人识别号：9141010037748287YM
- 地址、电话：北京市通州区高井文化产业园8号 010-82354236
- 开户行及账

In [15]:

img_prompt="""
请按照如下格式解析图片数据,可能会有一些模糊的内容你需要仔细查看，对于图片中的中文内容你需要仔细解析
**发票头部信息：**
- 发票代码
- 机器编号
- 发票名称
- 发票号码
- 开票日期

**购买方信息：**
- 名 称
- 纳税人识别号
- 地址、电话
- 开户行及账号

**密码区（密文）：**


**货物或应税劳务、服务明细：**

| 货物或应税劳务、服务名称 | 规格型号 | 单位 | 数量 | 单价 | 金额 | 税率 | 税额 |


**合计：**
- 金额
- 税额

**价税合计：**
- （大写
- （小写）

**销售方信息：**
- 名 称：
- 纳税人识别号：
- 地址、电话：
- 开户行及账号：

**签章及经办人信息：**
- 收款人：
- 复核：
- 开票人：
- 销售方：

**备注栏：**

**右侧边栏：**
第三联

**其他标识：**
- 右上角重复打印
- 销售方发票专用章内含税号
"""

prompt = f"""
请将这个图片的内容尽可能全部提取出来,之后进项校验，
提取规则如下
{img_prompt}
校验要求如下
    - 开票日期是否是当前时间的半年内
    - 每个条目：
      - 数量 * 单价 是否等于金额
      - 金额 * 税率 是否等于税额
    - 每个条目的金额之和 是否等于 合计行的金额
    - 每个条目的税额之和 是否等于 合计行的税额
    - 合计行的金额 + 税额 是否等于 价税合计行的小写
    - 价税合计行的大写金额 是否等于 小写金额
    - 本张发票的真假

你需要给我的结果是
1.所有解析到的发票信息
2.每个校验的结果和 数据
"""

messages = [{
    "role": "user",
    "content": [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
    ]
}]

response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=messages
)
print(response.choices[0].message.content)

 **发票信息解析结果：**

**发票头部信息：**
- 发票代码：1100213130
- 机器编号：928805152312
- 发票名称：北京增值税专用发票
- 发票号码：58971153
- 开票日期：2021年06月06日

**购买方信息：**
- 名 称：北京车无忧汽车修理服务有限公司
- 纳税人识别号：911101121024632563
- 地址、电话：北京市通州区新华南路85号 010-82351310
- 开户行及账号：工商银行北京通州区新华南路支行 0202011509102435

**密码区（密文）：**
```
>* 06 04 / 52 / + 90* + 80 99 > 3795 * 67
039 > 7 / 1062 < > < * * 13 + 7> 71 * < +32
4 </ - 21 / + 82018 < 30031 - +55 + 7 >> +
< 69 > 90 / 04 / 52 / + 90* + 80 99 >3795>
```

**货物或应税劳务、服务明细：**

| 货物或应税劳务、服务名称 | 规格型号 | 单位 | 数量 | 单价 | 金额 | 税率 | 税额 |
|---|---|---|---|---|---|---|---|
| *洗涤剂*汽车泡沫清洗剂 | | 件 | 10 | 16.00 | 160.00 | 13% | 20.80 |
| *日用杂品*汽车美容抛光用品 | | 套 | 5 | 150.00 | 750.00 | 13% | 97.50 |
| *日用杂品*汽车清洗用品 | | 套 | 5 | 60.00 | 300.00 | 13% | 39.00 |

**合计：**
- 金额：¥1,210.00
- 税额：¥157.30

**价税合计：**
- （大写）：壹仟叁佰陆拾柒圆叁角整
- （小写）：¥1367.30

**销售方信息：**
- 名 称：北京威客汽车用品旗舰店
- 纳税人识别号：9141010037748287TM
- 地址、电话：北京市通州区高井文化产业园8号 010-82354238
- 开户行及账号：工商银行北京燕莎东支行 0202011285624363333

**签章及经办人信息：**
- 收款人：惊也
- 复核：好懂
- 开票人：飘溢
- 销